# <p style="padding:10px;background-color:#87CEEB ;margin:10;color:#000000;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 10px 10px ;overflow:hidden;font-weight:50">1. Model Training for New York City Taxi Trip Duration Dataset</p>

### Loading the necessary Libraries

In [62]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Render plots directly below notebook cells
%matplotlib inline

# Suppress standard output warnings
import warnings
warnings.filterwarnings('ignore')

# Modelling
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import RandomizedSearchCV

# <p style="padding:10px;background-color:#87CEEB ;margin:10;color:#000000;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 10px 10px ;overflow:hidden;font-weight:50">2. Read the Dataset</p>

In [48]:
# 1. Load the training data with optimized types
file_path = r"D:\all\Python\ML Projects\New York City Taxi Trip Duration\notebook\data\train.csv"
optimized_dtypes = {
    'vendor_id': 'int8', 'passenger_count': 'int8',
    'pickup_longitude': 'float32', 'pickup_latitude': 'float32',
    'dropoff_longitude': 'float32', 'dropoff_latitude': 'float32',
    'store_and_fwd_flag': 'category', 'trip_duration': 'int32'
}

df_train = pd.read_csv(
    file_path, 
    dtype=optimized_dtypes, 
    parse_dates=['pickup_datetime', 'dropoff_datetime']
)





# # 5. Split into local train and validation sets (80% train, 20% validation)
# X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# print(f"Training features shape: {X_train.shape}")
# print(f"Validation features shape: {X_val.shape}")

### Feature Engineering: Temporal Extraction & Data Cleaning

In [49]:
# 2. Extract temporal features
df_train['pickup_hour'] = df_train['pickup_datetime'].dt.hour
df_train['pickup_dayofweek'] = df_train['pickup_datetime'].dt.dayofweek
df_train['pickup_month'] = df_train['pickup_datetime'].dt.month

# 3. Drop columns that cause leakage or cannot be processed
df_train = df_train.drop(columns=['id', 'pickup_datetime', 'dropoff_datetime'])

In [50]:
df_train.head(3)

,vendor_id,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration,pickup_hour,pickup_dayofweek,pickup_month
0,2,1,-73.982155,40.767937,-73.964630,40.765602,N,455,17,0,3
1,1,1,-73.980415,40.738564,-73.999481,40.731152,N,663,0,6,6
2,2,1,-73.979027,40.763939,-74.005333,40.710087,N,2124,11,1,1


# <p style="padding:10px;background-color:#87CEEB ;margin:10;color:#000000;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 10px 10px ;overflow:hidden;font-weight:50">Getting X and Y variables</p>

In [51]:
# 4. Define Features (X) and Target (y)
X = df_train.drop(columns=['trip_duration'])

# Apply the log transformation to the target variable to optimize for RMSLE
y = np.log1p(df_train['trip_duration'])

In [52]:
X.head(3)

,vendor_id,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,pickup_hour,pickup_dayofweek,pickup_month
0,2,1,-73.982155,40.767937,-73.964630,40.765602,N,17,0,3
1,1,1,-73.980415,40.738564,-73.999481,40.731152,N,0,6,6
2,2,1,-73.979027,40.763939,-74.005333,40.710087,N,11,1,1


# <p style="padding:10px;background-color:#87CEEB ;margin:10;color:#000000;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 10px 10px ;overflow:hidden;font-weight:50">Creating Data Transformation Pipeline</p>

### Creating Pipeline with Column Transformer

In [53]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Explicitly define column groups to ensure correct processing
categorical_cols = ['vendor_id', 'store_and_fwd_flag']
numerical_cols = [
    'passenger_count', 
    'pickup_longitude', 'pickup_latitude', 
    'dropoff_longitude', 'dropoff_latitude',
    'pickup_hour', 'pickup_dayofweek', 'pickup_month'
]

# 2. Numerical Pipeline: Impute missing values (if any) and scale distances/times
num_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())                
    ]
)

# 3. Categorical Pipeline: Impute missing values and One-Hot Encode
cat_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        # handle_unknown='ignore' prevents crashes if the Kaggle test set introduces a weird value
        ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
    ]
)

# 4. Combine both pipelines into the master preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num_pipeline', num_pipeline, numerical_cols),
        ('cat_pipeline', cat_pipeline, categorical_cols)
    ],
    remainder='drop' # Drops any columns not explicitly listed above (like ID or datetime strings if you forgot to drop them)
)

# Preview the preprocessor architecture in Jupyter
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_pipeline', ...), ('cat_pipeline', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` an

# <p style="padding:10px;background-color:#87CEEB ;margin:10;color:#000000;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 10px 10px ;overflow:hidden;font-weight:50">Train Test Split</p>

In [54]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

# <p style="padding:10px;background-color:#87CEEB ;margin:10;color:#000000;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 10px 10px ;overflow:hidden;font-weight:50">Transforming the data with pipeline created</p>

In [55]:
X_train = pd.DataFrame(preprocessor.fit_transform(X_train),columns=preprocessor.get_feature_names_out())
X_test = pd.DataFrame(preprocessor.transform(X_test),columns=preprocessor.get_feature_names_out())
preprocessor.get_feature_names_out()

array(['num_pipeline__passenger_count', 'num_pipeline__pickup_longitude',
       'num_pipeline__pickup_latitude', 'num_pipeline__dropoff_longitude',
       'num_pipeline__dropoff_latitude', 'num_pipeline__pickup_hour',
       'num_pipeline__pickup_dayofweek', 'num_pipeline__pickup_month',
       'cat_pipeline__vendor_id_2', 'cat_pipeline__store_and_fwd_flag_Y'],
      dtype=object)

In [56]:
X_train.head(3)

,num_pipeline__passenger_count,num_pipeline__pickup_longitude,num_pipeline__pickup_latitude,num_pipeline__dropoff_longitude,num_pipeline__dropoff_latitude,num_pipeline__pickup_hour,num_pipeline__pickup_dayofweek,num_pipeline__pickup_month,cat_pipeline__vendor_id_2,cat_pipeline__store_and_fwd_flag_Y
0,-0.50537,1.451712,0.535981,1.076158,-0.141096,0.686663,-0.537046,1.477586,1.0,0.0
1,-0.50537,0.202906,0.967390,-0.027191,0.250651,-0.094304,-1.560479,0.287999,1.0,0.0
2,-0.50537,0.053205,1.035657,-0.207986,0.551979,-0.250497,0.998104,0.882792,1.0,0.0


In [57]:
X_test.head(3)

,num_pipeline__passenger_count,num_pipeline__pickup_longitude,num_pipeline__pickup_latitude,num_pipeline__dropoff_longitude,num_pipeline__dropoff_latitude,num_pipeline__pickup_hour,num_pipeline__pickup_dayofweek,num_pipeline__pickup_month,cat_pipeline__vendor_id_2,cat_pipeline__store_and_fwd_flag_Y
0,-0.505370,-0.158496,-0.447959,-0.090670,0.239995,-1.031464,-0.537046,1.477586,1.0,0.0
1,-0.505370,-0.063895,0.402187,-0.239323,-0.053419,-0.250497,1.509821,0.287999,1.0,0.0
2,2.539686,-0.203696,-0.195210,0.000832,-0.085704,-1.812430,1.509821,1.477586,1.0,0.0


# <p style="padding:10px;background-color:#87CEEB ;margin:10;color:#000000;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 10px 10px ;overflow:hidden;font-weight:50">Model Training Baseline models</p>

### Create an Evaluate Function to give all metrics after model Training

In [58]:
# def evaluate_model(true_log, predicted_log):
#     # 1. Calculate Kaggle's metric (RMSLE)
#     # Since the inputs are already in log-space, standard RMSE here equals RMSLE
#     rmsle = root_mean_squared_error(true_log, predicted_log)
    
#     # 2. Reverse the log transformation to get actual seconds
#     true_real = np.expm1(true_log)
#     predicted_real = np.expm1(predicted_log)
    
#     # 3. Calculate interpretable human metrics in raw seconds
#     mae = mean_absolute_error(true_real, predicted_real)
#     rmse = root_mean_squared_error(true_real, predicted_real)
#     r2_square = r2_score(true_real, predicted_real)
    
#     return rmsle, mae, rmse, r2_square

In [59]:
def evaluate_model(true_log, predicted_log):
    # 1. Calculate stable metrics in the log-space (where the model trained)
    rmsle = root_mean_squared_error(true_log, predicted_log)
    r2_log = r2_score(true_log, predicted_log)
    
    # 2. Reverse the log transformation to get actual seconds for error tracking
    true_real = np.expm1(true_log)
    predicted_real = np.expm1(predicted_log)
    
    # 3. Calculate human-interpretable metrics in raw seconds
    mae = mean_absolute_error(true_real, predicted_real)
    rmse = root_mean_squared_error(true_real, predicted_real)
    
    return rmsle, mae, rmse, r2_log

### Training Various models

In [60]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "Decision Tree": DecisionTreeRegressor(),
    "XGBRegressor": XGBRegressor(n_jobs=-1), 
    "CatBoosting Regressor": CatBoostRegressor(verbose=False, thread_count=-1),
    "LGB Regressor": LGBMRegressor(n_jobs=-1)
    
    # --- WARNING: These models are computationally extreme for 1.45M rows ---
    # "K-Neighbors Regressor": KNeighborsRegressor(),
    # "Random Forest Regressor": RandomForestRegressor(n_jobs=-1),
    # "AdaBoost Regressor": AdaBoostRegressor(),
}

model_list = []
r2_list = []

# Using X_train and y_train as defined earlier (adjust if your variables are named xtrain/ytrain)
for i in range(len(list(models))):
    model = list(models.values())[i]
    model_name = list(models.keys())[i]
    
    # Train model
    model.fit(X_train, y_train) 

    # Make predictions
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_test)
    
    # Evaluate Train and Validation dataset (Unpacking all 4 metrics)
    train_rmsle, train_mae, train_rmse, train_r2 = evaluate_model(y_train, y_train_pred)
    val_rmsle, val_mae, val_rmse, val_r2 = evaluate_model(y_test, y_val_pred)
    
    print(model_name)
    model_list.append(model_name)
    
    print('Model performance for Training set')
    print("- Kaggle RMSLE Score: {:.4f}".format(train_rmsle))
    print("- Root Mean Squared Error (seconds): {:.4f}".format(train_rmse))
    print("- Mean Absolute Error (seconds): {:.4f}".format(train_mae))
    print("- R2 Score: {:.4f}".format(train_r2))

    print('----------------------------------')
    
    print('Model performance for Validation set')
    print("- Kaggle RMSLE Score: {:.4f}".format(val_rmsle))
    print("- Root Mean Squared Error (seconds): {:.4f}".format(val_rmse))
    print("- Mean Absolute Error (seconds): {:.4f}".format(val_mae))
    print("- R2 Score: {:.4f}".format(val_r2))
    
    r2_list.append(val_r2)
    
    print('='*35)
    print('\n')

Linear Regression
Model performance for Training set
- Kaggle RMSLE Score: 0.7783
- Root Mean Squared Error (seconds): 15352895635433402.0000
- Mean Absolute Error (seconds): 14216659422059.6094
- R2 Score: 0.0428
----------------------------------
Model performance for Validation set
- Kaggle RMSLE Score: 0.7763
- Root Mean Squared Error (seconds): 322878080394.7512
- Mean Absolute Error (seconds): 597799878.3445
- R2 Score: 0.0503


Lasso
Model performance for Training set
- Kaggle RMSLE Score: 0.7955
- Root Mean Squared Error (seconds): 5633.8989
- Mean Absolute Error (seconds): 566.8344
- R2 Score: 0.0000
----------------------------------
Model performance for Validation set
- Kaggle RMSLE Score: 0.7966
- Root Mean Squared Error (seconds): 3270.0105
- Mean Absolute Error (seconds): 568.0288
- R2 Score: -0.0000


Ridge
Model performance for Training set
- Kaggle RMSLE Score: 0.7783
- Root Mean Squared Error (seconds): 15352685119371936.0000
- Mean Absolute Error (seconds): 14216464

### Results

In [61]:
df_results = pd.DataFrame(list(zip(model_list, r2_list)), 
 columns=['Model Name', 'R2_Score']).sort_values(by=["R2_Score"],ascending=False)
df_results

,Model Name,R2_Score
5,CatBoosting Regressor,0.708205
4,XGBRegressor,0.689754
6,LGB Regressor,0.613345
3,Decision Tree,0.430488
2,Ridge,0.050308
0,Linear Regression,0.050308
1,Lasso,-0.000003


# <p style="padding:10px;background-color:#87CEEB ;margin:10;color:#000000;font-family:newtimeroman;font-size:100%;text-align:center;border-radius: 10px 10px ;overflow:hidden;font-weight:50">Hyperparameter tuning</p>

### Tuning Catboost

In [63]:


# 1. Initialize CatBoost (using thread_count=-1 to use all CPU cores)
cbr = CatBoostRegressor(verbose=False, thread_count=-1)

# 2. Hyperparameter grid
# (Your ranges are excellent for CatBoost)
param_dist = {
    'depth': [4, 5, 6, 7, 8, 9, 10],
    'learning_rate': [0.01, 0.02, 0.03, 0.04],
    'iterations': [300, 400, 500, 600]
}

# 3. Instantiate RandomizedSearchCV object
# CHANGED: cv=3 (down from 5) to save massive compute time
# CHANGED: n_iter=10 (explicitly controlling how many random combinations to try)
# CHANGED: scoring='neg_root_mean_squared_error' to optimize for the Kaggle RMSLE metric directly
rscv = RandomizedSearchCV(
    estimator=cbr, 
    param_distributions=param_dist, 
    n_iter=10, 
    scoring='neg_root_mean_squared_error', 
    cv=3, 
    n_jobs=-1, 
    random_state=42
)

# 4. Fit the model 
# CRUCIAL: Ensure you are using the processed pipeline data (e.g., X_train_processed)
rscv.fit(X_train, y_train)

# 5. Print the tuned parameters and score
print("Best Parameters:", rscv.best_params_)

# Multiply by -1 because scikit-learn makes the RMSE negative for optimization purposes
print("Best CV RMSLE Score:", -rscv.best_score_)

Best Parameters: {'learning_rate': 0.04, 'iterations': 600, 'depth': 6}
Best CV RMSLE Score: 0.46823941478421865


### Definition to print evaluated model results

In [64]:
def print_evaluated_results(model, X_train, y_train, X_test, y_test):
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Evaluate Train and Test dataset (unpacking all 4 metrics)
    train_rmsle, train_mae, train_rmse, train_r2 = evaluate_model(y_train, y_train_pred)
    test_rmsle, test_mae, test_rmse, test_r2 = evaluate_model(y_test, y_test_pred)

    # Printing results
    print('Model performance for Training set')
    print("- Kaggle RMSLE Score: {:.4f}".format(train_rmsle))
    print("- Root Mean Squared Error (seconds): {:.4f}".format(train_rmse))
    print("- Mean Absolute Error (seconds): {:.4f}".format(train_mae))
    print("- R2 Score: {:.4f}".format(train_r2))

    print('----------------------------------')
    
    print('Model performance for Test set')
    print("- Kaggle RMSLE Score: {:.4f}".format(test_rmsle))
    print("- Root Mean Squared Error (seconds): {:.4f}".format(test_rmse))
    print("- Mean Absolute Error (seconds): {:.4f}".format(test_mae))
    print("- R2 Score: {:.4f}".format(test_r2))

In [65]:
# Selecting best model
best_cbr = rscv.best_estimator_

# Evaluate Train and Test dataset
print_evaluated_results(best_cbr,X_train,y_train,X_test,y_test)

Model performance for Training set
- Kaggle RMSLE Score: 0.4666
- Root Mean Squared Error (seconds): 5596.3313
- Mean Absolute Error (seconds): 336.6809
- R2 Score: 0.6561
----------------------------------
Model performance for Test set
- Kaggle RMSLE Score: 0.4693
- Root Mean Squared Error (seconds): 3206.1936
- Mean Absolute Error (seconds): 338.0431
- R2 Score: 0.6530
